In [10]:
import psycopg2
from faker import Faker
import random
from datetime import date, timedelta, datetime

fake = Faker()
random.seed(42)
Faker.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# DATABASE CONFIGURATION  ← update before running
# ─────────────────────────────────────────────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "abc_foodmart",
    "user":     "postgres",
    "password": "123",
}


def get_conn():
    return psycopg2.connect(**DB_CONFIG)


def random_date(start: date, end: date) -> date:
    return start + timedelta(days=random.randint(0, (end - start).days))

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN 1 – WORKFORCE
# ─────────────────────────────────────────────────────────────────────────────

def insert_stores(cur):
    stores = [
        ("ABC Foodmart - Flushing",
         "136-20 Roosevelt Ave, Flushing, NY 11354",
         "Queens",   "Active",  date(1994,  3, 15)),
        ("ABC Foodmart - Jackson Heights",
         "74-05 37th Ave, Jackson Heights, NY 11372",
         "Queens",   "Active",  date(1997,  6, 22)),
        ("ABC Foodmart - Bay Ridge",
         "8612 5th Ave, Bay Ridge, NY 11209",
         "Brooklyn", "Active",  date(2025,  1, 10)),
        ("ABC Foodmart - Flatbush",
         "1402 Flatbush Ave, Brooklyn, NY 11210",
         "Brooklyn", "Pending", date(2025,  4,  1)),
        ("ABC Foodmart - Bushwick",
         "1090 Broadway, Bushwick, NY 11221",
         "Brooklyn", "Pending", date(2025,  6,  1)),
    ]
    cur.executemany(
        """INSERT INTO Stores (store_name, address, borough, status, opened_date)
           VALUES (%s,%s,%s,%s,%s)""", stores)
    print(f"  ✓ {len(stores)} stores")


def insert_departments(cur):
    depts = [
        ("Produce",),
        ("Dairy",),
        ("Bakery",),
        ("Meat & Seafood",),
        ("Deli",),
        ("Frozen Foods",),
        ("Snacks & Confectionery",),
        ("Canned & Dry Goods",),
        ("Household",),
        ("Management",),
    ]
    cur.executemany(
        "INSERT INTO Departments (department_name) VALUES (%s)",
        depts
    )
    print(f"✓ {len(depts)} departments")


ROLE_DEPARTMENT_MAP = {
    "Cashier": ["Management"], 
    "Stock Clerk": ["Produce", "Dairy", "Bakery", "Frozen Foods",
                    "Snacks & Confectionery", "Canned & Dry Goods", "Household"],
    "Produce Associate": ["Produce"],
    "Butcher": ["Meat & Seafood"],
    "Deli Associate": ["Deli"],
    "Department Manager": ["Produce", "Dairy", "Bakery", "Meat & Seafood", "Deli"],
    "Shift Supervisor": ["Management"],
    "Assistant Manager": ["Management"],
    "Store Manager": ["Management"],
}

def insert_employees(cur):
    cur.execute("SELECT store_id FROM Stores ORDER BY store_id")
    store_ids = [r[0] for r in cur.fetchall()]

    # Department name → id
    cur.execute("SELECT department_id, department_name FROM Departments")
    dept_map = {name: did for did, name in cur.fetchall()}

    rows = []

    for sid in store_ids:

        # ─────────────────────────────────────────────
        # 1. CORE MANAGEMENT (fixed structure)
        # ─────────────────────────────────────────────
        rows.append((
            sid, dept_map["Management"],
            fake.first_name(), fake.last_name(),
            "Store Manager",
            round(random.uniform(28, 35), 2),
            "09:00:00", "17:00:00",
            random_date(date(2015,1,1), date(2023,1,1)),
            "Full-Time"
        ))

        for _ in range(random.randint(1, 2)):
            rows.append((
                sid, dept_map["Management"],
                fake.first_name(), fake.last_name(),
                "Assistant Manager",
                round(random.uniform(22, 30), 2),
                "09:00:00", "17:00:00",
                random_date(date(2016,1,1), date(2024,1,1)),
                "Full-Time"
            ))

        for _ in range(random.randint(1, 2)):
            shift = random.choice(["morning", "afternoon"])
            rows.append((
                sid, dept_map["Management"],
                fake.first_name(), fake.last_name(),
                "Shift Supervisor",
                round(random.uniform(20, 26), 2),
                "06:00:00" if shift == "morning" else "14:00:00",
                "14:00:00" if shift == "morning" else "22:00:00",
                random_date(date(2018,1,1), date(2025,1,1)),
                "Full-Time"
            ))

        # ─────────────────────────────────────────────
        # 2. DEPARTMENT STAFF (1 per key dept)
        # ─────────────────────────────────────────────
        dept_roles = [
            ("Produce", "Produce Associate"),
            ("Meat & Seafood", "Butcher"),
            ("Deli", "Deli Associate"),
        ]

        for dept_name, role in dept_roles:
            rows.append((
                sid, dept_map[dept_name],
                fake.first_name(), fake.last_name(),
                role,
                round(random.uniform(17, 24), 2),
                "06:00:00", "14:00:00",
                random_date(date(2019,1,1), date(2025,1,1)),
                "Full-Time"
            ))

        # ─────────────────────────────────────────────
        # 3. STOCK CLERKS (spread across departments)
        # ─────────────────────────────────────────────
        stock_departments = ROLE_DEPARTMENT_MAP["Stock Clerk"]

        for _ in range(random.randint(4, 7)):
            dept_name = random.choice(stock_departments)
            shift = random.choice(["morning", "afternoon"])

            rows.append((
                sid, dept_map[dept_name],
                fake.first_name(), fake.last_name(),
                "Stock Clerk",
                round(random.uniform(16, 20), 2),
                "06:00:00" if shift == "morning" else "14:00:00",
                "14:00:00" if shift == "morning" else "22:00:00",
                random_date(date(2020,1,1), date(2025,1,1)),
                random.choice(["Full-Time", "Part-Time"])
            ))

        # ─────────────────────────────────────────────
        # 4. CASHIERS (front of store)
        # ─────────────────────────────────────────────
        for _ in range(random.randint(5, 8)):
            shift = random.choice(["morning", "afternoon"])

            rows.append((
                sid, dept_map["Management"],
                fake.first_name(), fake.last_name(),
                "Cashier",
                round(random.uniform(16, 19), 2),
                "06:00:00" if shift == "morning" else "14:00:00",
                "14:00:00" if shift == "morning" else "22:00:00",
                random_date(date(2021,1,1), date(2025,1,1)),
                random.choice(["Part-Time", "Full-Time"])
            ))

    cur.executemany("""
        INSERT INTO Employees
        (store_id, department_id, first_name, last_name, role,
         hourly_rate, shift_start, shift_end, hire_date, employment_type)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, rows)

    print(f"✓ {len(rows)} employees")


# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN 2 – PRODUCTS & INVENTORY
# ─────────────────────────────────────────────────────────────────────────────

def insert_product_categories(cur):
    cats = [
        ("Fresh Produce",), ("Dairy & Eggs",), ("Bakery",), ("Meat",),
        ("Seafood",), ("Frozen Foods",), ("Beverages",), ("Snacks",),
        ("Canned Goods",), ("Pasta & Rice",), ("Condiments & Sauces",),
        ("Personal Care",), ("Household Cleaning",), ("Deli & Prepared Foods",),
        ("Organic",), ("International Foods",), ("Baby Products",),
        ("Pet Supplies",), ("Coffee & Tea",), ("Breakfast Cereals",),
    ]
    cur.executemany(
        "INSERT INTO ProductCategories (category_name) VALUES (%s)", cats
    )
    print(f"✓ {len(cats)} product categories")


# Product catalog
_PRODUCTS = {
    "Fresh Produce": [
        ("Organic Bananas", 0.45, 0.99),
        ("Fuji Apples", 0.60, 1.49),
        ("Baby Spinach 5oz", 1.20, 2.99),
        ("Roma Tomatoes", 0.55, 1.29),
        ("Russet Potatoes 5lb", 2.50, 4.99),
        ("Avocados", 0.80, 1.79),
        ("Broccoli Crown", 0.90, 1.99),
        ("Yellow Onions 3lb", 1.20, 2.49),
    ],
    "Dairy & Eggs": [
        ("Whole Milk 1 Gal", 3.20, 5.49),
        ("Large Eggs 12ct", 2.80, 4.99),
        ("Cheddar Cheese 8oz", 2.50, 4.29),
        ("Greek Yogurt 32oz", 3.10, 5.79),
        ("Salted Butter 1lb", 4.00, 6.49),
        ("Cream Cheese 8oz", 1.80, 3.29),
    ],
    "Bakery": [
        ("White Sandwich Bread", 1.80, 3.49),
        ("Whole Wheat Bread", 2.00, 3.99),
        ("Bagels 6ct", 2.50, 4.49),
        ("Croissants 4ct", 3.00, 5.49),
        ("Sourdough Loaf", 3.50, 6.99),
    ],
    "Meat": [
        ("Ground Beef 1lb", 4.00, 7.99),
        ("Chicken Breast 2lb", 5.50, 9.99),
        ("Pork Chops 1lb", 4.20, 7.49),
        ("Italian Sausage 1lb", 3.80, 6.99),
        ("Ribeye Steak 1lb", 10.00, 18.99),
    ],
    "Seafood": [
        ("Atlantic Salmon Fillet", 8.00, 14.99),
        ("Tilapia Fillet 1lb", 5.00, 9.49),
        ("Fresh Shrimp 1lb", 7.50, 13.99),
        ("Canned Tuna 5oz", 0.80, 1.79),
    ],
    "Frozen Foods": [
        ("Frozen Pizza 12in", 3.50, 6.99),
        ("Ice Cream 1qt", 2.80, 5.49),
        ("Frozen Broccoli 12oz", 1.20, 2.49),
        ("Frozen Waffles 10ct", 2.50, 4.99),
        ("Frozen Burritos 8ct", 4.00, 7.49),
    ],
    "Beverages": [
        ("Orange Juice 64oz", 2.80, 5.49),
        ("Spring Water 24pk", 5.00, 8.99),
        ("Cola 2L", 1.20, 2.49),
        ("Green Tea Bottled", 1.50, 2.99),
    ],
    "Coffee & Tea": [
        ("Ground Coffee 12oz", 5.00, 9.99),
        ("Coffee K-Cups 12ct", 6.00, 11.99),
        ("Black Tea 20ct", 2.00, 3.99),
        ("Espresso Pods 10ct", 6.00, 11.49),
    ],
    "Snacks": [
        ("Potato Chips 8oz", 2.00, 3.99),
        ("Mixed Nuts 10oz", 4.50, 8.99),
        ("Granola Bars 6ct", 2.80, 5.49),
        ("Pretzels 16oz", 2.00, 3.79),
        ("Dark Chocolate Bar", 1.80, 3.49),
    ],
    "Canned Goods": [
        ("Tomato Sauce 24oz", 1.00, 1.99),
        ("Black Beans 15oz", 0.70, 1.49),
        ("Chicken Broth 32oz", 1.50, 2.99),
        ("Corn 15oz", 0.60, 1.29),
        ("Diced Tomatoes 14.5oz", 0.80, 1.69),
    ],
    "Pasta & Rice": [
        ("Spaghetti 1lb", 1.00, 1.99),
        ("Penne 1lb", 1.00, 1.99),
        ("Jasmine Rice 5lb", 4.00, 7.49),
        ("Basmati Rice 5lb", 4.50, 8.49),
        ("Mac & Cheese Box", 0.80, 1.59),
    ],
    "Condiments & Sauces": [
        ("Ketchup 32oz", 1.80, 3.49),
        ("Yellow Mustard 14oz", 1.20, 2.29),
        ("Soy Sauce 10oz", 1.50, 2.99),
        ("Olive Oil 17oz", 4.50, 8.99),
        ("Ranch Dressing 16oz", 2.00, 3.99),
    ],
    "Personal Care": [
        ("Shampoo 12oz", 3.00, 5.99),
        ("Body Wash 18oz", 2.50, 4.99),
        ("Toothpaste 4oz", 1.80, 3.49),
        ("Deodorant 2.6oz", 2.50, 4.99),
    ],
    "Household Cleaning": [
        ("Dish Soap 22oz", 2.00, 3.99),
        ("Paper Towels 6ct", 5.00, 9.99),
        ("Laundry Detergent 50oz", 6.00, 11.99),
        ("Trash Bags 30ct", 4.00, 7.99),
    ],
    "Deli & Prepared Foods": [
        ("Turkey Slices 8oz", 3.50, 6.99),
        ("Ham Slices 8oz", 3.00, 5.99),
        ("Swiss Cheese 8oz", 3.20, 5.79),
        ("Rotisserie Chicken", 6.00, 10.99),
    ],
}

# SKU prefix mapping (cleaner than slicing names)
PREFIX_MAP = {
    "Fresh Produce": "PRO",
    "Dairy & Eggs": "DAI",
    "Bakery": "BAK",
    "Meat": "MEA",
    "Seafood": "SEA",
    "Frozen Foods": "FRO",
    "Beverages": "BEV",
    "Snacks": "SNK",
    "Canned Goods": "CAN",
    "Pasta & Rice": "PAS",
    "Condiments & Sauces": "CON",
    "Personal Care": "PER",
    "Household Cleaning": "HOU",
    "Deli & Prepared Foods": "DEL",
    "Coffee & Tea": "COF",
}


def insert_products(cur):
    cur.execute("SELECT category_id, category_name FROM ProductCategories")
    cat_map = {name: cid for cid, name in cur.fetchall()}

    rows = []
    ctr = 1

    for cat_name, items in _PRODUCTS.items():
        cat_id = cat_map[cat_name]
        prefix = PREFIX_MAP[cat_name]

        for pname, standard_cost, retail_price in items:

            if cat_name in ["Fresh Produce", "Meat", "Seafood"]:
                brand_type = "National"
            else:
                brand_type = random.choice(["National", "Store Brand"])

            if cat_name in ["Fresh Produce", "Dairy & Eggs"]:
                reorder = random.randint(20, 60)
            elif cat_name in ["Canned Goods", "Pasta & Rice"]:
                reorder = random.randint(50, 150)
            else:
                reorder = random.randint(15, 80)

            rows.append((
                cat_id,
                f"{prefix}-{ctr:04d}",
                pname,
                brand_type,
                round(standard_cost, 2),
                retail_price,
                reorder
            ))
            ctr += 1

    cur.executemany("""
        INSERT INTO Products
        (category_id, sku, product_name, brand_type,
         unit_cost, retail_price, reorder_threshold)
        VALUES (%s,%s,%s,%s,%s,%s,%s)
    """, rows)

    print(f"✓ {len(rows)} products inserted")


def insert_inventory(cur):
    cur.execute("""
        SELECT p.product_id, p.reorder_threshold, c.category_name
        FROM Products p
        JOIN ProductCategories c ON p.category_id = c.category_id
    """)
    products = cur.fetchall()

    cur.execute("SELECT store_id FROM Stores")
    store_ids = [r[0] for r in cur.fetchall()]

    rows = []

    for sid in store_ids:
        for pid, reorder, cat in products:

            # 20% low stock (for analytics)
            if random.random() < 0.20:
                qty = random.randint(0, max(0, reorder - 1))

            else:
                if cat in ["Fresh Produce", "Dairy & Eggs", "Meat", "Seafood"]:
                    qty = random.randint(20, 200)
                elif cat in ["Canned Goods", "Pasta & Rice"]:
                    qty = random.randint(200, 1000)
                else:
                    qty = random.randint(50, 400)

            rows.append((sid, pid, qty))

    cur.executemany("""
        INSERT INTO Inventory (store_id, product_id, quantity_on_hand)
        VALUES (%s,%s,%s)
    """, rows)

    print(f"✓ {len(rows)} inventory records inserted")


# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN 3 – PROCUREMENT
# ─────────────────────────────────────────────────────────────────────────────
def insert_vendors(cur):
    vendors = [
        ("FreshPeak Produce Co.",    "James Rivera",   "jrivera@freshpeak.com",     "718-555-0101", "Net 30"),
        ("Garden State Farms",       "Linda Park",     "lpark@gsfarms.com",         "201-555-0202", "Net 15"),
        ("Metro Dairy Distributors", "Carlos Mendez",  "cmendez@metrodairy.com",    "212-555-0303", "Net 30"),
        ("Sunrise Bakery Supply",    "Patricia Wong",  "pwong@sunrisebakery.com",   "718-555-0404", "Net 7"),
        ("Premier Meat Packers",     "Robert Johnson", "rjohnson@premiermeats.com", "631-555-0505", "Net 14"),
        ("Atlantic Seafood Dist.",   "Maria Gonzalez", "mgonzalez@atlanticsfd.com", "718-555-0606", "Net 7"),
        ("FreezeWay Foods",          "David Kim",      "dkim@freezewayfoods.com",   "347-555-0707", "Net 30"),
        ("BevPro Beverages",         "Susan Lee",      "slee@bevpro.com",           "516-555-0808", "Net 30"),
        ("SnackWorld Inc.",          "Thomas Brown",   "tbrown@snackworld.com",     "718-555-0909", "Net 21"),
        ("Pantry Essentials Corp.",  "Angela Davis",   "adavis@pantryess.com",      "212-555-1010", "Net 30"),
    ]

    cur.executemany("""
        INSERT INTO Vendors
        (vendor_name, contact_name, contact_email, contact_phone, payment_terms)
        VALUES (%s,%s,%s,%s,%s)
    """, vendors)

    print(f"✓ {len(vendors)} vendors inserted")

VENDOR_CATEGORY_MAP = {
    "FreshPeak Produce Co.": ["Fresh Produce"],
    "Garden State Farms": ["Fresh Produce", "Organic"],
    "Metro Dairy Distributors": ["Dairy & Eggs"],
    "Sunrise Bakery Supply": ["Bakery"],
    "Premier Meat Packers": ["Meat"],
    "Atlantic Seafood Dist.": ["Seafood"],
    "FreezeWay Foods": ["Frozen Foods"],
    "BevPro Beverages": ["Beverages", "Coffee & Tea"],
    "SnackWorld Inc.": ["Snacks"],
    "Pantry Essentials Corp.": ["Canned Goods", "Pasta & Rice", "Condiments & Sauces"],
}

def insert_vendor_products(cur):
    """
    Each vendor supplies only products in allowed categories.
    Ensures:
    - category realism
    - multi-sourcing of products
    - full coverage of the catalog
    """

    cur.execute("SELECT vendor_id, vendor_name FROM Vendors")
    vendors = cur.fetchall()

    cur.execute("""
        SELECT p.product_id, c.category_name
        FROM Products p
        JOIN ProductCategories c ON p.category_id = c.category_id
    """)
    products = cur.fetchall()

    pairs = set()

    for vid, vname in vendors:
        allowed = VENDOR_CATEGORY_MAP.get(vname, [])
        
        valid_products = [
            pid for pid, cat in products
            if cat in allowed
        ]

        if not valid_products:
            continue

        chosen = random.sample(
            valid_products,
            min(len(valid_products), random.randint(5, 15))
        )

        for pid in chosen:
            pairs.add((vid, pid))

    # Ensure every product has at least one vendor
    all_product_ids = [p[0] for p in products]
    vendor_ids = [v[0] for v in vendors]

    for pid in all_product_ids:
        if not any(pid == p[1] for p in pairs):
            pairs.add((random.choice(vendor_ids), pid))

    cur.executemany("""
        INSERT INTO VendorProducts (vendor_id, product_id)
        VALUES (%s,%s)
    """, list(pairs))

    print(f"✓ {len(pairs)} vendor-product relationships inserted")

def generate_procurement_cost(unit_cost, category_name):
    """
    Generates realistic procurement price based on product unit_cost.
    Keeps your category logic but anchors to product cost.
    """

    variability = {
        "Fresh Produce": (0.75, 1.10),
        "Dairy & Eggs": (0.85, 1.10),
        "Bakery": (0.80, 1.15),
        "Meat": (0.90, 1.15),
        "Seafood": (0.90, 1.20),
        "Frozen Foods": (0.85, 1.10),
        "Beverages": (0.80, 1.15),
        "Snacks": (0.80, 1.15),
        "Canned Goods": (0.75, 1.10),
        "Pasta & Rice": (0.75, 1.10),
        "Condiments & Sauces": (0.80, 1.15),
        "Coffee & Tea": (0.85, 1.20),
    }

    lo, hi = variability.get(category_name, (0.85, 1.15))

    return round(unit_cost * random.uniform(lo, hi), 2)

def insert_purchase_orders_and_items(cur):

    cur.execute("SELECT store_id FROM Stores")
    store_ids = [r[0] for r in cur.fetchall()]

    cur.execute("SELECT vendor_id FROM Vendors")
    vendor_ids = [r[0] for r in cur.fetchall()]

    # ✅ FIXED: include unit_cost
    cur.execute("""
        SELECT p.product_id, p.unit_cost, c.category_name
        FROM Products p
        JOIN ProductCategories c ON p.category_id = c.category_id
    """)
    product_data = {
        pid: (cost, cat)
        for pid, cost, cat in cur.fetchall()
    }

    cur.execute("SELECT product_id FROM Products")
    product_ids = [r[0] for r in cur.fetchall()]

    statuses = ["Pending", "Shipped", "Delivered", "Cancelled"]
    weights = [0.10, 0.10, 0.75, 0.05]

    po_item_records = []

    for store_id in store_ids:

        for _ in range(random.randint(120, 140)):

            vendor_id = random.choice(vendor_ids)

            order_date = random_date(date(2024,1,1), date(2025,3,31))
            expected_delivery = order_date + timedelta(days=random.randint(1,7))
            status = random.choices(statuses, weights=weights)[0]

            cur.execute("""
                INSERT INTO PurchaseOrders
                (store_id, vendor_id, order_date, expected_delivery, status)
                VALUES (%s,%s,%s,%s,%s)
                RETURNING po_id
            """, (store_id, vendor_id, order_date, expected_delivery, status))

            po_id = cur.fetchone()[0]

            chosen_products = random.sample(product_ids, random.randint(1, 4))

            for pid in chosen_products:

                unit_cost, category = product_data[pid]

                # ✅ FIXED: anchored cost
                purchase_price = generate_procurement_cost(unit_cost, category)

                qty = random.randint(20, 200)

                cur.execute("""
                    INSERT INTO PurchaseOrderItems
                    (po_id, product_id, quantity, unit_purchase_price)
                    VALUES (%s,%s,%s,%s)
                    RETURNING po_item_id
                """, (po_id, pid, qty, purchase_price))

                po_item_id = cur.fetchone()[0]

                po_item_records.append((
                    po_item_id,
                    pid,
                    expected_delivery,
                    store_id,
                    status,
                    qty
                ))

    print(f"✓ Procurement complete: {len(po_item_records)} line items generated")

    return po_item_records

def insert_deliveries(cur, po_item_meta):

    cur.execute("SELECT employee_id FROM Employees ORDER BY employee_id")
    emp_ids = [r[0] for r in cur.fetchall()]

    notes = [None, "Partial shipment", "Damaged packaging", "On time", "Slight delay"]
    note_weights = [0.70, 0.07, 0.07, 0.09, 0.07]

    rows = []

    for po_item_id, product_id, expected_delivery, store_id, status, ordered_qty in po_item_meta:

        # ✅ FIX: only valid delivery statuses
        if status not in ("Shipped", "Delivered"):
            continue

        # quantity variability
        received_qty = max(
            1,
            int(ordered_qty * random.uniform(0.85, 1.0))
        )

        # ✅ IMPROVED timing logic
        if status == "Delivered":
            day_offset = random.randint(-1, 2)
        else:  # Shipped
            day_offset = random.randint(0, 4)

        actual_delivery = expected_delivery + timedelta(days=day_offset)

        rows.append((
            po_item_id,
            received_qty,
            actual_delivery,
            random.choice(emp_ids),
            random.choices(notes, weights=note_weights)[0]
        ))

    cur.executemany("""
        INSERT INTO Deliveries
        (po_item_id, quantity_received, actual_delivery, received_by, notes)
        VALUES (%s,%s,%s,%s,%s)
    """, rows)

    print(f"✓ {len(rows)} deliveries inserted")

    
# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN 4 – SALES
# ─────────────────────────────────────────────────────────────────────────────

def insert_customers(cur, n=600):
    rows = []

    for _ in range(n):
        is_loyal = random.random() < 0.45

        rows.append((
            fake.first_name(),
            fake.last_name(),
            fake.email() if random.random() < 0.75 else None,
            is_loyal,
            random_date(date(2015,1,1), date(2025,3,1)) if is_loyal else None
        ))

    cur.executemany("""
        INSERT INTO Customers
        (first_name, last_name, email, loyalty_member, joined_date)
        VALUES (%s,%s,%s,%s,%s)
    """, rows)

    print(f"✓ {n} customers inserted")


def insert_promotions(cur):
    promotions = [
        ("Summer Sale", 15, date(2024,6,1), date(2024,6,30), None),
        ("Back to School", 10, date(2024,8,15), date(2024,9,15), None),
        ("Holiday Deals", 20, date(2024,11,25), date(2024,12,31), None),
        ("New Year Promo", 12, date(2025,1,1), date(2025,1,31), None),
        ("Spring Deals", 8, date(2025,3,1), date(2025,3,31), None),
        ("Loyalty Exclusive", 10, date(2024,9,1), date(2024,9,30), None),
    ]

    cur.executemany("""
        INSERT INTO Promotions
        (promotion_name, discount_pct, start_date, end_date, store_id)
        VALUES (%s,%s,%s,%s,%s)
    """, promotions)

    print(f"✓ {len(promotions)} promotions inserted")


def insert_promotion_products(cur):
    cur.execute("SELECT promotion_id FROM Promotions")
    promo_ids = [r[0] for r in cur.fetchall()]

    cur.execute("SELECT product_id FROM Products")
    product_ids = [r[0] for r in cur.fetchall()]

    pairs = set()

    for promo_id in promo_ids:
        sample_size = random.randint(5, 20)

        selected_products = random.sample(
            product_ids,
            min(sample_size, len(product_ids))
        )

        for product_id in selected_products:
            pairs.add((promo_id, product_id))

    cur.executemany("""
        INSERT INTO PromotionProducts (promotion_id, product_id)
        VALUES (%s,%s)
    """, list(pairs))

    print(f"✓ {len(pairs)} promotion-product links inserted")
    
def insert_transactions_and_items(cur):
    """
    Realistic sales generation aligned with schema + analytics goals:

    ✔ Only Active stores generate transactions
    ✔ 900–1100 transactions per store (volume realism)
    ✔ 2–8 items per transaction (basket size realism)
    ✔ Promotions applied ONLY if:
        - Active on transaction date
        - Product is part of the promotion
    ✔ ~25% of items get a promotion
    ✔ ~70% transactions linked to customers
    ✔ Inventory decremented via trigger
    ✔ total_amount computed via trigger
    """

    # ─────────────────────────────────────────────────────────
    # Load base data
    # ─────────────────────────────────────────────────────────
    cur.execute("SELECT store_id FROM Stores WHERE status='Active'")
    store_ids = [r[0] for r in cur.fetchall()]

    cur.execute("SELECT customer_id FROM Customers")
    cust_ids = [r[0] for r in cur.fetchall()]

    cur.execute("SELECT product_id, retail_price FROM Products")
    products = cur.fetchall()

    # Promotions with validity windows
    cur.execute("SELECT promotion_id, start_date, end_date FROM Promotions")
    promos = cur.fetchall()

    # Promotion → product mapping (IMPORTANT FIX)
    cur.execute("SELECT promotion_id, product_id FROM PromotionProducts")
    promo_product_map = {}
    for promo_id, product_id in cur.fetchall():
        promo_product_map.setdefault(promo_id, set()).add(product_id)

    pay_methods = ["Cash", "Credit", "Debit", "Mobile"]
    pay_weights = [0.15, 0.40, 0.35, 0.10]

    tx_start = date(2024, 1, 1)
    tx_end   = date(2025, 3, 31)

    total_tx = 0
    total_items = 0

    # ─────────────────────────────────────────────────────────
    # Generate transactions
    # ─────────────────────────────────────────────────────────
    for store_id in store_ids:

        for _ in range(random.randint(900, 1100)):

            customer_id = random.choice(cust_ids) if random.random() < 0.70 else None

            tx_date = random_date(tx_start, tx_end)
            tx_time = datetime(
                tx_date.year,
                tx_date.month,
                tx_date.day,
                random.randint(8, 21),
                random.randint(0, 59)
            )

            cur.execute("""
                INSERT INTO Transactions
                (store_id, customer_id, transaction_time, payment_method, total_amount)
                VALUES (%s,%s,%s,%s,0)
                RETURNING transaction_id
            """, (
                store_id,
                customer_id,
                tx_time,
                random.choices(pay_methods, weights=pay_weights)[0]
            ))

            tx_id = cur.fetchone()[0]
            total_tx += 1

            # ─────────────────────────────────────────
            # Transaction items
            # ─────────────────────────────────────────
            n_items = random.randint(2, 8)

            selected_products = random.sample(
                products,
                min(n_items, len(products))
            )

            for product_id, retail_price in selected_products:

                # Active promotions for this date
                valid_promos = [
                    promo_id
                    for promo_id, start, end in promos
                    if start <= tx_date <= end
                ]

                # FILTER promos that apply to THIS product (CRITICAL FIX)
                valid_promos = [
                    p for p in valid_promos
                    if product_id in promo_product_map.get(p, set())
                ]

                promo_id = (
                    random.choice(valid_promos)
                    if valid_promos and random.random() < 0.25
                    else None
                )

                cur.execute("""
                    INSERT INTO TransactionItems
                    (transaction_id, product_id, promotion_id,
                     quantity, unit_price_at_sale)
                    VALUES (%s,%s,%s,%s,%s)
                """, (
                    tx_id,
                    product_id,
                    promo_id,
                    random.randint(1, 3),
                    retail_price
                ))

                total_items += 1

    print(f"✓ {total_tx} transactions inserted")
    print(f"✓ {total_items} transaction items inserted")

# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN 5 – FINANCE
# ─────────────────────────────────────────────────────────────────────────────

def insert_expense_categories(cur):
    cats = [
        ("Payroll",),
        ("Rent",),
        ("Utilities",),
        ("Supplies",),
        ("Equipment Maintenance",),
        ("Marketing & Advertising",),
        ("Insurance",),
        ("Inventory Shrinkage",),
        ("Miscellaneous",),
    ]

    cur.executemany("""
        INSERT INTO ExpenseCategories (category_name)
        VALUES (%s)
    """, cats)

    print(f"✓ {len(cats)} expense categories inserted")

def insert_accounting_entries(cur):
    """
    Monthly expense entries per store (Jan 2024 – Mar 2025)

    Design choices:
    - Payroll excluded (handled in PayrollEntries → avoids double counting)
    - Fixed monthly expenses (Rent, Insurance) → low variability
    - Variable expenses (Utilities, Supplies, etc.) → higher variability
    - Supports:
        ✔ Q6: Gross profit
        ✔ Q11: Store performance
    """

    cur.execute("SELECT store_id FROM Stores")
    store_ids = [r[0] for r in cur.fetchall()]

    cur.execute("""
        SELECT category_id, category_name
        FROM ExpenseCategories
    """)
    exp_cats = cur.fetchall()

    # More realistic cost behavior by type
    ranges = {
        "Rent":                    (9000, 13000),   # stable
        "Utilities":               (1800, 3500),    # seasonal-ish
        "Supplies":                (500, 1800),     # variable
        "Equipment Maintenance":   (200, 1200),
        "Marketing & Advertising": (300, 1500),
        "Insurance":               (700, 1200),     # stable
        "Inventory Shrinkage":     (200, 900),
        "Miscellaneous":           (100, 600),
    }

    rows = []

    for sid in store_ids:

        period = date(2024, 1, 1)

        while period <= date(2025, 3, 1):

            for cat_id, cat_name in exp_cats:

                if cat_name == "Payroll":
                    continue  # handled separately

                lo, hi = ranges.get(cat_name, (100, 1000))

                # Slight seasonality for utilities (nice realism boost)
                multiplier = 1.0
                if cat_name == "Utilities" and period.month in [1, 2, 7, 8]:
                    multiplier = random.uniform(1.1, 1.3)

                amount = round(random.uniform(lo, hi) * multiplier, 2)

                rows.append((
                    sid,
                    cat_id,
                    "Expense",
                    amount,
                    period,
                    f"{cat_name} – {period.strftime('%b %Y')}",
                ))

            # Move to next month
            m, y = period.month, period.year
            period = date(y + (m // 12), (m % 12) + 1, 1)

    cur.executemany("""
        INSERT INTO AccountingEntries
        (store_id, category_id, entry_type, amount, period_date, description)
        VALUES (%s,%s,%s,%s,%s,%s)
    """, rows)

    print(f"✓ {len(rows)} accounting entries inserted")

def insert_payroll_entries(cur):
    """
    Bi-weekly payroll (Jan 2024 – Mar 2025)

    Improvements:
    - Hours tied loosely to employment type (Full-Time vs Part-Time)
    - Keeps realism without overcomplicating logic

    Supports:
        ✔ Q10: Payroll % of revenue by department
        ✔ Q6/Q11: Profitability
    """

    cur.execute("""
        SELECT employee_id, store_id, hourly_rate, employment_type
        FROM Employees
    """)
    employees = cur.fetchall()

    # Generate bi-weekly periods
    periods = []
    d = date(2024, 1, 15)

    while d <= date(2025, 3, 31):
        periods.append(d)
        d += timedelta(weeks=2)

    rows = []

    for emp_id, sid, hourly_rate, emp_type in employees:

        for period in periods:

            # More realistic hours
            if emp_type == "Full-Time":
                hours = random.uniform(70, 85)
            else:
                hours = random.uniform(25, 60)

            hours = round(hours, 2)

            gross_pay = round(hours * float(hourly_rate), 2)

            rows.append((
                emp_id,
                sid,
                period,
                hours,
                hourly_rate,
                gross_pay
            ))

    cur.executemany("""
        INSERT INTO PayrollEntries
        (employee_id, store_id, period_date, hours_worked, hourly_rate, gross_pay)
        VALUES (%s,%s,%s,%s,%s,%s)
    """, rows)

    print(f"✓ {len(rows)} payroll entries inserted")

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("\n🛒  ABC Foodmart – Data Loading (schema v5)")
    print("=" * 50)
    conn = get_conn()
    cur  = conn.cursor()
    try:
        print("\n[1/5] Workforce …")
        insert_stores(cur)
        insert_departments(cur)
        insert_employees(cur)
        conn.commit()

        print("\n[2/5] Products & Inventory …")
        insert_product_categories(cur)
        insert_products(cur)
        insert_inventory(cur)       # uses product_id — v5
        conn.commit()

        print("\n[3/5] Procurement …")
        insert_vendors(cur)
        insert_vendor_products(cur) # NEW — VendorProducts bridge
        po_item_meta = insert_purchase_orders_and_items(cur)  # NEW — PurchaseOrderItems
        insert_deliveries(cur, po_item_meta)  # uses po_item_id — v5
        conn.commit()

        print("\n[4/5] Sales …")
        insert_customers(cur)
        insert_promotions(cur)
        insert_promotion_products(cur)  # NEW — PromotionProducts bridge
        cur.execute("ALTER TABLE TransactionItems DISABLE TRIGGER trg_decrement_inventory;")
        insert_transactions_and_items(cur)
        cur.execute("ALTER TABLE TransactionItems ENABLE TRIGGER trg_decrement_inventory;")
        conn.commit()

        print("\n[5/5] Finance …")
        insert_expense_categories(cur)
        insert_accounting_entries(cur)
        insert_payroll_entries(cur)
        conn.commit()

        print("\n✅  All data loaded successfully!")
        print("\n── Quick row-count checks ──────────────────────────")
        for tbl in ["Stores","Departments","Employees","ProductCategories",
                    "Products","Inventory","Vendors","VendorProducts",
                    "PurchaseOrders","PurchaseOrderItems","Deliveries",
                    "Customers","Promotions","PromotionProducts",
                    "Transactions","TransactionItems",
                    "ExpenseCategories","AccountingEntries","PayrollEntries"]:
            cur.execute(f"SELECT COUNT(*) FROM {tbl}")
            print(f"  {tbl:<22} {cur.fetchone()[0]:>7} rows")

    except Exception as exc:
        conn.rollback()
        print(f"\n❌  Error: {exc}")
        raise
    finally:
        cur.close()
        conn.close()

if __name__ == "__main__":
    main()


🛒  ABC Foodmart – Data Loading (schema v5)

[1/5] Workforce …
  ✓ 5 stores
  ✓ 12 departments
  ✓ 89 employees

[2/5] Products & Inventory …
  ✓ 20 product categories
  ✓ 91 products
  ✓ 455 inventory records (89 below reorder threshold — supports Q3)

[3/5] Procurement …
  ✓ 15 vendors
  ✓ 159 vendor-product links
  ✓ 1622 purchase order items across ~650 POs
  ✓ 1407 deliveries (mix of on-time & late — supports Q5)

[4/5] Sales …
  ✓ 600 customers (45% loyalty members — supports Q7)
  ✓ 10 promotions
  ✓ 109 promotion-product links (supports Q4)
  ✓ 3021 transactions, 15197 transaction items
    (2–8 items/txn supports Q8 market basket; 25% promo rate supports Q4)

[5/5] Finance …
  ✓ 9 expense categories
  ✓ 600 accounting entries (supports Q6, Q11)
  ✓ 2848 payroll entries (bi-weekly × 89 employees — supports Q10)

✅  All data loaded successfully!

── Quick row-count checks ──────────────────────────
  Stores                       5 rows
  Departments                 12 rows
  Emp

In [13]:

TABLES = [
    ("Stores",           "SELECT * FROM Stores"),
    ("Departments",      "SELECT * FROM Departments"),
    ("Employees",        "SELECT * FROM Employees LIMIT 5"),
    ("Products",         "SELECT * FROM Products LIMIT 5"),
    ("Inventory",        "SELECT * FROM Inventory LIMIT 5"),
    ("Vendors",          "SELECT * FROM Vendors LIMIT 5"),
    ("Customers",        "SELECT * FROM Customers LIMIT 5"),
    ("Transactions",     "SELECT * FROM Transactions LIMIT 5"),
    ("TransactionItems", "SELECT * FROM TransactionItems LIMIT 5"),
]

conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

for label, query in TABLES:
    cur.execute(query)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print("  " + " | ".join(f"{c:<20}" for c in cols))
    print("  " + "-" * (23 * len(cols)))
    for row in rows:
        print("  " + " | ".join(f"{str(v):<20}" for v in row))

cur.close()
conn.close()



────────────────────────────────────────────────────────────
  Stores
────────────────────────────────────────────────────────────
  store_id             | store_name           | address              | borough              | status               | opened_date         
  ------------------------------------------------------------------------------------------------------------------------------------------
  1                    | ABC Foodmart - Flushing | 136-20 Roosevelt Ave, Flushing, NY 11354 | Queens               | Active               | 1994-03-15          
  2                    | ABC Foodmart - Jackson Heights | 74-05 37th Ave, Jackson Heights, NY 11372 | Queens               | Active               | 1997-06-22          
  3                    | ABC Foodmart - Bay Ridge | 8612 5th Ave, Bay Ridge, NY 11209 | Brooklyn             | Active               | 2025-01-10          
  4                    | ABC Foodmart - Flatbush | 1402 Flatbush Ave, Brooklyn, NY 11210 | Brooklyn     